Loading the Libraries


In [1]:
import h5py
import pandas as pd
import numpy as np
import os

file paths

In [15]:

# labels = pd.read_csv("/Users/vaidehipandya/Desktop/Work/HPC_labels_subset.csv")
# latent = pd.read_csv("/Users/vaidehipandya/Desktop/Work/exports/train_img_h_latent.csv")


# ==== CONFIG: INPUT FILES FOR KB CLEANING ====

BASE_DIR = "/nfs/home/users/vpandya"     

# Main representation file
H5_FILE = f"{BASE_DIR}/data/SHARED/1 Week/Kai-Rakovic-M42/Vaidehi/hdf5_TCGA_LUAD_5x_he_train_reps.h5"


# Label file
TILE_LABEL_FILE = f"{BASE_DIR}/TCGA_LUAD_5x_he_train_filtered_leiden_2p5__fold2.csv"

#Survival file
SURVIVAL = f"{BASE_DIR}/all_folds_summary.csv"



In [8]:


# Load label file
labels = pd.read_csv(TILE_LABEL_FILE)

# Load survival file
survival = pd.read_csv(SURVIVAL)

Extract all datasets

In [16]:
OUTPUT_DIR = "./h5_exports"  # where CSVs will be saved

os.makedirs(OUTPUT_DIR, exist_ok=True)

with h5py.File(H5_FILE, "r") as f:
    
    print("Datasets found:", list(f.keys()))
    
    for key in f.keys():
        data = f[key][:]
        
        print(f"Processing dataset: {key} with shape {data.shape}")
        
        # If dataset is 1D → simple CSV
        if data.ndim == 1:
            df = pd.DataFrame({key: data})
        
        # If dataset is 2D → treat rows as embeddings / arrays
        elif data.ndim == 2:
            df = pd.DataFrame(data)
        
        else:
            # If it is weird shape, flatten safely
            df = pd.DataFrame(data.reshape(data.shape[0], -1))

        # Convert bytes → strings if needed
        df = df.applymap(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x)

        # Save CSV
        out_path = os.path.join(OUTPUT_DIR, f"{key}.csv")
        df.to_csv(out_path, index=False)
        
        print(f"Saved → {out_path}")

print("Done! All datasets extracted.")

Datasets found: ['train_hist_subtype', 'train_img_h_latent', 'train_img_z_latent', 'train_labels', 'train_patterns', 'train_samples', 'train_slides', 'train_tiles']
Processing dataset: train_hist_subtype with shape (826977,)


/tmp/ipykernel_3363189/1585916732.py:27: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x)


Saved → ./h5_exports/train_hist_subtype.csv
Processing dataset: train_img_h_latent with shape (826977, 1024)
Saved → ./h5_exports/train_img_h_latent.csv
Processing dataset: train_img_z_latent with shape (826977, 128)
Saved → ./h5_exports/train_img_z_latent.csv
Processing dataset: train_labels with shape (826977,)
Saved → ./h5_exports/train_labels.csv
Processing dataset: train_patterns with shape (826977,)
Saved → ./h5_exports/train_patterns.csv
Processing dataset: train_samples with shape (826977,)
Saved → ./h5_exports/train_samples.csv
Processing dataset: train_slides with shape (826977,)
Saved → ./h5_exports/train_slides.csv
Processing dataset: train_tiles with shape (826977,)
Saved → ./h5_exports/train_tiles.csv
Done! All datasets extracted.


In [19]:


# Clean column names
labels.columns = (
    labels.columns.str.strip()
                   .str.lower()
                   .str.replace(" ", "_")
                   .str.replace("-", "_")
                   .str.replace(".", "_")
)

# Rename leiden column to hpc_id (if present)
if "leiden_2_5" in labels.columns:
    labels = labels.rename(columns={"leiden_2_5": "hpc_id"})
elif "leiden_2.5" in labels.columns:
    labels = labels.rename(columns={"leiden_2.5": "hpc_id"})

# Rename structural columns
labels = labels.rename(columns={
    "samples": "sample_id",
    "slides": "slide_id",
    "tiles": "tile_name"
})

# Clean whitespace in key columns
for col in ["sample_id", "slide_id", "tile_name"]:
    if col in labels.columns:
        labels[col] = labels[col].astype(str).str.strip()

print("Labels cleaned:")
print(labels.head())
print(labels.columns.tolist())

Labels cleaned:
      sample_id                 slide_id   tile_name  hpc_id
0  TCGA-55-7574  TCGA-55-7574-01Z-00-DX1  18_15.jpeg      32
1  TCGA-55-7574  TCGA-55-7574-01Z-00-DX1   14_6.jpeg      50
2  TCGA-55-7574  TCGA-55-7574-01Z-00-DX1  11_23.jpeg      27
3  TCGA-55-7574  TCGA-55-7574-01Z-00-DX1  10_20.jpeg      21
4  TCGA-55-7574  TCGA-55-7574-01Z-00-DX1   23_1.jpeg       6
['sample_id', 'slide_id', 'tile_name', 'hpc_id']


h_latent_for_kb

In [20]:


# Load latent file
latent = pd.read_csv("/nfs/home/users/vpandya/h5_exports/train_img_h_latent.csv")

# Rename latent columns
latent.columns = [f"latent_{i}" for i in range(latent.shape[1])]

# Merge side-by-side
merged = pd.concat([labels.reset_index(drop=True), latent.reset_index(drop=True)], axis=1)
print("Merged successfully!")
print("Shape:", merged.shape)

# Clean whitespace
for col in ["samples", "slides", "tiles"]:
    if col in merged.columns:
        merged[col] = merged[col].astype(str).str.strip()

# Rename leiden_2.5 → hpc_id BEFORE auto detection
if "leiden_2.5" in merged.columns:
    merged = merged.rename(columns={"leiden_2.5": "hpc_id"})

# Auto-detect hpc column
hpc_col = None
for c in merged.columns:
    if "hpc" in c.lower():      # now picks up hpc_id directly
        hpc_col = c
        break

if hpc_col is None:
    raise KeyError("Could not find a column with 'hpc' in its name.")

# Final hpc_id column
merged["hpc_id"] = merged[hpc_col].astype(str).str.strip()

# Rename columns for DB consistency
merged = merged.rename(columns={
    "samples": "sample_id",
    "slides": "slide_id",
    "tiles": "tile_name"
})

# Add image_index column
merged["image_index"] = range(len(merged))

# Add h5 source path column
merged["h5_source_path"] = "/data/SHARED/1 Week/Kai-Rakovic-M42/Vaidehi/hdf5_TCGA_LUAD_5x_he_train_reps.h5"

# Save final file
merged.to_csv("h_latent_full.csv", index=False)
print("Saved: h_latent_full.csv")
print("Final shape:", merged.shape)
print("Columns:", merged.columns[:15].tolist(), "...")

Merged successfully!
Shape: (826977, 1028)
Saved: h_latent_full.csv
Final shape: (826977, 1030)
Columns: ['sample_id', 'slide_id', 'tile_name', 'hpc_id', 'latent_0', 'latent_1', 'latent_2', 'latent_3', 'latent_4', 'latent_5', 'latent_6', 'latent_7', 'latent_8', 'latent_9', 'latent_10'] ...


Classifier Integration

In [ ]:

# Clean column names
survival.columns = (
    survival.columns.str.strip()
                    .str.lower()
                    .str.replace(" ", "_")
                    .str.replace("(", "")
                    .str.replace(")", "")
                    .str.replace("%", "")
                    .str.replace("-", "_")
)

# Rename covariate to hpc_id
if "covariate" in survival.columns:
    survival = survival.rename(columns={"covariate": "hpc_id"})
else:
    print("No column named 'covariate' found. Check column names:")
    print(survival.columns)

# Fix "-log2p" to "log2p"
if "-log2p" in survival.columns:
    survival = survival.rename(columns={"-log2p": "log2p"})

# Save cleaned file
survival.to_csv("survival.csv", index=False)

print("✔ Saved cleaned survival.csv")
print(survival.head())

✔ Saved cleaned survival.csv
   hpc_id      coef   expcoef    secoef  coef_lower_95  coef_upper_95  \
0  HPC 56 -0.090552  0.913527  0.057940      -0.204145       0.023042   
1  HPC 60 -0.049650  0.951660  0.065954      -0.179169       0.079869   
2   HPC 0 -0.048440  0.952749  0.031415      -0.110150       0.013270   
3  HPC 59 -0.047115  0.954245  0.072809      -0.189828       0.095598   
4  HPC 69 -0.029432  0.971035  0.060150      -0.147430       0.088566   

   expcoef_lower_95  expcoef_upper_95         z         p    _log2p        se  
0          0.815490          1.023357 -1.558394  0.017988  3.098820  0.057957  
1          0.836305          1.082996 -0.752971  0.617810  1.170443  0.066083  
2          0.895883          1.013241 -1.536970  0.020767  3.036461  0.031485  
3          0.827342          1.100617 -0.647344  0.725957  1.009104  0.072814  
4          0.863060          1.092546 -0.490292  0.903346  0.694296  0.060204  


Tile_registry

In [8]:
# Load your CSV


labels.rename(columns={'leiden_2.5': 'hpc_id'}, inplace=True)

# Clean column names
labels.columns = labels.columns.str.strip().str.lower()

# If any missing columns, fill them in
labels["image_index"] = range(len(labels))
labels["h5_source_path"] = "/data/SHARED/1 Week/Kai-Rakovic-M42/Vaidehi/hdf5_TCGA_LUAD_5x_he_train_reps.h5"

# Save cleaned file
labels.to_csv("tile_reg_full.csv", index=False)
print(" Saved tile_reg_full.csv")

NameError: name 'labels' is not defined

In [9]:
import pandas as pd

df = pd.read_csv("/Users/vaidehipandya/Desktop/Work/csv_files/new.csv/tile_reg.csv")

df.columns = df.columns.str.strip()
df["slides"] = df["slides"].astype(str).str.strip().str.upper()
df["tiles"]  = df["tiles"].astype(str).str.strip()

In [10]:
df["slide_tile"] = df["slides"] + "_" + df["tiles"]

In [11]:
dupes = (
    df.groupby("slide_tile")
      .size()
      .reset_index(name="n")
      .query("n > 1")
)
print("Total rows:", len(df))
print("Duplicate slide_tile rows:", len(dupes))
dupes.head(20)

Total rows: 360667
Duplicate slide_tile rows: 100


,slide_tile,n
19309,TCGA-49-4488-01Z-00-DX1_12_12.jpeg,2
19310,TCGA-49-4488-01Z-00-DX1_12_13.jpeg,2
19321,TCGA-49-4488-01Z-00-DX1_13_11.jpeg,2
19322,TCGA-49-4488-01Z-00-DX1_13_12.jpeg,2
19323,TCGA-49-4488-01Z-00-DX1_13_13.jpeg,2
19337,TCGA-49-4488-01Z-00-DX1_14_16.jpeg,2
19350,TCGA-49-4488-01Z-00-DX1_15_16.jpeg,2
19373,TCGA-49-4488-01Z-00-DX1_17_17.jpeg,2
19374,TCGA-49-4488-01Z-00-DX1_17_18.jpeg,2
19395,TCGA-49-4488-01Z-00-DX1_19_22.jpeg,2


In [19]:
tile = pd.read_csv("/Users/vaidehipandya/Desktop/Work/loaded_data/TCGA_LUAD_5x_he_train_filtered_leiden_2p5__fold2.csv")
tile.columns = tile.columns.str.strip()
tile["slides"] = tile["slides"].astype(str).str.strip().str.upper()
tile["tiles"]  = tile["tiles"].astype(str).str.strip()
tile["slide_tile"] = tile["slides"] + "_" + tile["tiles"]
dupes = (
    tile.groupby("slide_tile")
      .size()
      .reset_index(name="n")
      .query("n > 1")
)
print("Total rows:", len(df))
print("Duplicate slide_tile rows:", len(dupes))
dupes.head(20)


Total rows: 360667
Duplicate slide_tile rows: 0


,slide_tile,n


KeyError: "Column(s) ['image_index'] do not exist"

In [15]:
total_rows = len(df)
unique_tiles = df["slide_tile"].nunique()

dupe_keys = df["slide_tile"].duplicated(keep=False).sum()

conflicts = (
    df.groupby("slide_tile")
      .agg(
          n=("slide_tile", "size"),
          uniq_image_index=("image_index", "nunique"),
          uniq_hpc_id=("hpc_id", "nunique"),
      )
      .reset_index()
)

conflict_keys = conflicts.query("n > 1 and (uniq_image_index > 1 or uniq_hpc_id > 1)")

print("Total rows:", total_rows)
print("Unique slide_tile:", unique_tiles)
print("Rows involved in duplicates:", dupe_keys)
print("Conflicting duplicate keys:", len(conflict_keys))

Total rows: 360667
Unique slide_tile: 360567
Rows involved in duplicates: 200
Conflicting duplicate keys: 100


In [16]:
dupe_rows = df[df["slide_tile"].duplicated(keep=False)].copy()
dupe_rows.to_csv("tile_reg_dup.csv", index=False)
print("Saved backup:", len(dupe_rows))

Saved backup: 200


In [17]:
df = df.copy()
df["image_index"] = pd.to_numeric(df["image_index"], errors="coerce")

df_clean = (
    df.sort_values(["slide_tile", "image_index"])
      .drop_duplicates("slide_tile", keep="first")
)

print("Before:", len(df), "After:", len(df_clean))
print("Unique slide_tile after:", df_clean["slide_tile"].is_unique)

Before: 360667 After: 360567
Unique slide_tile after: True


In [25]:
# Clean column names


# If any missing columns, fill them in
tile["image_index"] = range(len(tile))
tile["h5_source_path"] = "/nfs/long-term-scratch/Vaidehi/TCGA/hdf5_TCGA_LUAD_5x_he_train_reps.h5"

# Save cleaned file
tile.to_csv("tile_reg.csv", index=False)


In [24]:
tile = pd.read_csv("/Users/vaidehipandya/Desktop/Work/tile_reg.csv")

dupes = (
    tile.groupby("slide_tile")
      .size()
      .reset_index(name="n")
      .query("n > 1")
)
print("Total rows:", len(df))
print("Duplicate slide_tile rows:", len(dupes))
dupes.head(20)


Total rows: 360667
Duplicate slide_tile rows: 0


,slide_tile,n
